# Build Gold Layer

## Create Gold Database

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
print("Gold database created")

Gold database created


## Load Silver Tables & Build Master

In [0]:
from pyspark.sql.functions import (
    col, sum, count, countDistinct, 
    avg, round, year, month, date_format
)

# Load silver tables
orders_enriched       = spark.table("silver.orders_enriched")
order_items_enriched  = spark.table("silver.order_items_enriched")

# Master table — join both on OrderID
master = order_items_enriched.join(
    orders_enriched,
    on="OrderID",
    how="left"
)

print(f"Master table: {master.count()} rows")
print(f"Columns: {master.columns}")

Master table: 199 rows
Columns: ['OrderID', 'ProductID', 'OrderDetailID', 'Quantity', 'ProductName', 'Unit', 'Price', 'CategoryName', 'Description', 'SupplierName', 'SupplierCity', 'SupplierCountry', 'LineRevenue', 'OrderDate', 'CustomerName', 'ContactName', 'CustomerCity', 'CustomerCountry', 'EmployeeName', 'ShipperName']


## Build Gold Table: Sales Overview

In [0]:
gold_sales_overview = (
    master
    .withColumn("Year", year(col("OrderDate")))
    .withColumn("Month", month(col("OrderDate")))
    .withColumn("YearMonth", date_format(col("OrderDate"), "yyyy-MM"))
    .groupBy("Year", "Month", "YearMonth")
    .agg(
        round(sum("LineRevenue"), 2).alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders"),
        sum("Quantity").alias("TotalItemsSold"),
        round(avg("LineRevenue"), 2).alias("AvgOrderValue")
    )
    .orderBy("YearMonth")
)

(gold_sales_overview.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.sales_overview"))

print(f"gold.sales_overview: {gold_sales_overview.count()} rows")
gold_sales_overview.show(3)

gold.sales_overview: 4 rows
+----+-----+---------+------------+-----------+--------------+-------------+
|Year|Month|YearMonth|TotalRevenue|TotalOrders|TotalItemsSold|AvgOrderValue|
+----+-----+---------+------------+-----------+--------------+-------------+
|2024|   11|  2024-11|     20651.0|         12|           736|       625.79|
|2024|   12|  2024-12|    29396.85|         22|          1261|       534.49|
|2025|    1|  2025-01|    36028.29|         23|          1355|       562.94|
+----+-----+---------+------------+-----------+--------------+-------------+
only showing top 3 rows


## Build Gold Table: Product & Category Performance

In [0]:
gold_product_performance = (
    master
    .groupBy("ProductID", "ProductName", "CategoryName", "Unit", "Price")
    .agg(
        sum("Quantity").alias("TotalQuantitySold"),
        round(sum("LineRevenue"), 2).alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders")
    )
    .orderBy(col("TotalRevenue").desc())
)

(gold_product_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.product_performance"))

print(f"gold.product_performance: {gold_product_performance.count()} rows")
gold_product_performance.show(3)

gold.product_performance: 67 rows
+---------+--------------------+--------------+--------------------+------+-----------------+------------+-----------+
|ProductID|         ProductName|  CategoryName|                Unit| Price|TotalQuantitySold|TotalRevenue|TotalOrders|
+---------+--------------------+--------------+--------------------+------+-----------------+------------+-----------+
|       59|Raclette Courdavault|Dairy Products|           5 kg pkg.|  55.0|              201|     11055.0|          7|
|       62|      Tarte au sucre|   Confections|             48 pies|  49.3|              192|      9465.6|          8|
|       29|Thüringer Rostbra...|  Meat/Poultry|50 bags x 30 sausgs.|123.79|               50|      6189.5|          3|
+---------+--------------------+--------------+--------------------+------+-----------------+------------+-----------+
only showing top 3 rows


## Build Gold Table 3: Customer Geography

In [0]:
gold_customer_geography = (
    master
    .groupBy("CustomerName", "CustomerCity", "CustomerCountry")
    .agg(
        round(sum("LineRevenue"), 2).alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders"),
        sum("Quantity").alias("TotalItemsBought")
    )
    .orderBy(col("TotalRevenue").desc())
)

(gold_customer_geography.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.customer_geography"))

print(f"gold.customer_geography: {gold_customer_geography.count()} rows")
gold_customer_geography.show(3)

gold.customer_geography: 45 rows
+--------------------+------------+---------------+------------+-----------+----------------+
|        CustomerName|CustomerCity|CustomerCountry|TotalRevenue|TotalOrders|TotalItemsBought|
+--------------------+------------+---------------+------------+-----------+----------------+
|Rattlesnake Canyo...| Albuquerque|            USA|     11420.4|          5|             349|
|          QUICK-Stop|   Cunewalde|        Germany|      9406.3|          4|             425|
|    Suprêmes délices|   Charleroi|        Belgium|      8051.3|          2|             185|
+--------------------+------------+---------------+------------+-----------+----------------+
only showing top 3 rows


## Build Gold Table 4: Employee Performance

In [0]:
gold_employee_performance = (
    master
    .groupBy("EmployeeName")
    .agg(
        round(sum("LineRevenue"), 2).alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders"),
        sum("Quantity").alias("TotalItemsSold"),
        round(avg("LineRevenue"), 2).alias("AvgRevenuePerOrder")
    )
    .orderBy(col("TotalRevenue").desc())
)

(gold_employee_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.employee_performance"))

print(f"gold.employee_performance: {gold_employee_performance.count()} rows")
gold_employee_performance.show(3)

gold.employee_performance: 9 rows
+----------------+------------+-----------+--------------+------------------+
|    EmployeeName|TotalRevenue|TotalOrders|TotalItemsSold|AvgRevenuePerOrder|
+----------------+------------+-----------+--------------+------------------+
|Margaret Peacock|    25509.44|         16|           839|            554.55|
|  Laura Callahan|     19638.8|         12|           573|            654.63|
|   Nancy Davolio|     18652.2|         11|           623|            666.15|
+----------------+------------+-----------+--------------+------------------+
only showing top 3 rows


# Build Gold Table 5: Shipper Performance

In [0]:
gold_shipper_performance = (
    master
    .groupBy("ShipperName")
    .agg(
        round(sum("LineRevenue"), 2).alias("TotalRevenue"),
        countDistinct("OrderID").alias("TotalOrders"),
        sum("Quantity").alias("TotalItemsShipped")
    )
    .orderBy(col("TotalRevenue").desc())
)

(gold_shipper_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.shipper_performance"))

print(f"gold.shipper_performance: {gold_shipper_performance.count()} rows")
gold_shipper_performance.show()

gold.shipper_performance: 3 rows
+----------------+------------+-----------+-----------------+
|     ShipperName|TotalRevenue|TotalOrders|TotalItemsShipped|
+----------------+------------+-----------+-----------------+
|  United Package|    43805.65|         31|             1630|
|Federal Shipping|    36975.85|         25|             1371|
|  Speedy Express|    31576.79|         21|             1199|
+----------------+------------+-----------+-----------------+



# Verify All Gold Tables

In [0]:
gold_tables = [
    "sales_overview",
    "product_performance",
    "customer_geography",
    "employee_performance",
    "shipper_performance"
]

for table in gold_tables:
    df = spark.table(f"gold.{table}")
    print(f"gold.{table}: {df.count()} rows")
    print(f"   Columns: {df.columns}\n")

gold.sales_overview: 4 rows
   Columns: ['Year', 'Month', 'YearMonth', 'TotalRevenue', 'TotalOrders', 'TotalItemsSold', 'AvgOrderValue']

gold.product_performance: 67 rows
   Columns: ['ProductID', 'ProductName', 'CategoryName', 'Unit', 'Price', 'TotalQuantitySold', 'TotalRevenue', 'TotalOrders']

gold.customer_geography: 45 rows
   Columns: ['CustomerName', 'CustomerCity', 'CustomerCountry', 'TotalRevenue', 'TotalOrders', 'TotalItemsBought']

gold.employee_performance: 9 rows
   Columns: ['EmployeeName', 'TotalRevenue', 'TotalOrders', 'TotalItemsSold', 'AvgRevenuePerOrder']

gold.shipper_performance: 3 rows
   Columns: ['ShipperName', 'TotalRevenue', 'TotalOrders', 'TotalItemsShipped']

